# Modeling

In [43]:
import os
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold, cross_val_score, GridSearchCV, cross_val_predict, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (roc_auc_score, f1_score, precision_score,
                             recall_score, make_scorer, fbeta_score)
from sklearn.calibration import CalibratedClassifierCV
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.base import clone

from xgboost import XGBClassifier

In [44]:
DATA_DIR = r"D:\Ameng\Data Science Project\heart-failure-prediction\data"
df = pd.read_csv(os.path.join(DATA_DIR, "heart_failure_clinical_records_dataset.csv"))

In [45]:
X = df.drop("DEATH_EVENT", axis=1)
y = df["DEATH_EVENT"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (299, 12)
y shape: (299,)


In [46]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

X_train: (239, 12)
X_test: (60, 12)


In [47]:
print("Train target distribution:")
print(y_train.value_counts())

print("\nTrain target proportion:")
print(y_train.value_counts(normalize=True))

print("\nTest target distribution:")
print(y_test.value_counts())

print("\nTest target proportion:")
print(y_test.value_counts(normalize=True))

Train target distribution:
DEATH_EVENT
0    162
1     77
Name: count, dtype: int64

Train target proportion:
DEATH_EVENT
0    0.677824
1    0.322176
Name: proportion, dtype: float64

Test target distribution:
DEATH_EVENT
0    41
1    19
Name: count, dtype: int64

Test target proportion:
DEATH_EVENT
0    0.683333
1    0.316667
Name: proportion, dtype: float64


In [48]:
def build_preprocessor():
    log_cols = ["creatinine_phosphokinase", "serum_creatinine", "platelets", "time"]
    scale_cols = ["age", "ejection_fraction", "serum_sodium"]
    bin_cols = ["anaemia", "diabetes", "high_blood_pressure", "sex", "smoking"]

    log_pipeline = Pipeline([
        ("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
        ("scale", StandardScaler())
    ])
    
    scale_pipeline = Pipeline([
        ("scale", StandardScaler())
    ])

    return ColumnTransformer([
        ("log", log_pipeline, log_cols),
        ("scale", scale_pipeline, scale_cols),
        ("pass", "passthrough", bin_cols)
    ], verbose_feature_names_out=False)

## Cross Validation

In [49]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scorers = {
    "AUC": make_scorer(roc_auc_score, response_method="predict_proba"),
    "F1": make_scorer(f1_score, zero_division=0),
    "PRE": make_scorer(precision_score, zero_division=0),
    "REC": make_scorer(recall_score, zero_division=0),
}

def cv_table(X, y, models):
    rows = []
    for name, model in models.items():
        r = {m: cross_val_score(model, X, y, cv=cv, scoring=s).mean().round(4)
             for m, s in scorers.items()}
        r |= {"model": name}; rows.append(r)
    return pd.DataFrame(rows).set_index("model")

In [50]:
preprocessor = build_preprocessor()

def make_model(est):
    return Pipeline([
        ("pre", preprocessor),
        ("model", est)
    ])

In [51]:
models = {
    "LogReg": make_model(LogisticRegression(max_iter=1000, class_weight="balanced", solver="liblinear")),
    "RandomForest": make_model(RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced")),
    "SVM": make_model(CalibratedClassifierCV(SVC(kernel="rbf", class_weight="balanced"), ensemble=False)),
    "kNN": make_model(KNeighborsClassifier(n_neighbors=5)),
    "GradientBoosting": make_model(GradientBoostingClassifier(random_state=42)),
}
result = cv_table(X_train, y_train, models)
print(result.round(4).sort_values("AUC", ascending=False))

                     AUC      F1     PRE     REC
model                                           
RandomForest      0.9078  0.7443  0.7424  0.7550
LogReg            0.9038  0.7932  0.7730  0.8175
GradientBoosting  0.8972  0.7036  0.7616  0.6642
SVM               0.8956  0.7477  0.7616  0.7400
kNN               0.8315  0.5399  0.8167  0.4167


In [52]:
top3_names = (result.sort_values("AUC", ascending=False).head(3).index.tolist())

print("Top 3 models:")
print(top3_names)

Top 3 models:
['RandomForest', 'LogReg', 'GradientBoosting']


Based on 5-fold cross-validation result, the top three models will be selected for hyperparameter tuning.

## Hyperparameter Tuning

In [53]:
param_lr = {
    "model__C": [0.001, 0.01, 0.1, 1, 10, 100],
    "model__penalty": ["l1", "l2"]
}

param_rf = {
    "model__n_estimators": [100, 200, 300],
    "model__max_depth": [None, 5, 10],
    "model__min_samples_split": [2, 5],
    "model__min_samples_leaf": [1, 2]
}

In [54]:
grid_lr = GridSearchCV(
    make_model(LogisticRegression(max_iter=1000, class_weight="balanced", solver="liblinear")),
    param_grid=param_lr,
    cv=cv, scoring="roc_auc", n_jobs=-1)
grid_lr.fit(X_train, y_train)
print("Best param:", grid_lr.best_params_)
print("Best CV AUC:", grid_lr.best_score_.round(4))

Best param: {'model__C': 0.1, 'model__penalty': 'l2'}
Best CV AUC: 0.9091


d:\Ameng\Data Science Project\heart-failure-prediction\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


In [55]:
grid_rf = GridSearchCV(
    make_model(RandomForestClassifier(random_state=42, class_weight="balanced")),
    param_grid=param_rf,
    cv=cv, scoring="roc_auc", n_jobs=-1)

grid_rf.fit(X_train, y_train)

print("Best param:", grid_rf.best_params_)
print("Best CV AUC:", grid_rf.best_score_.round(4))

Best param: {'model__max_depth': 10, 'model__min_samples_leaf': 2, 'model__min_samples_split': 2, 'model__n_estimators': 300}
Best CV AUC: 0.9154


## Tuned Model Comparison

In [56]:
tuned_results = []

if "LogReg" in top3_names:
    tuned_results.append({
        "Model": "LogReg",
        "Best CV AUC": grid_lr.best_score_,
        "Best Parameters": grid_lr.best_params_
    })

if "RandomForest" in top3_names:
    tuned_results.append({
        "Model": "RandomForest",
        "Best CV AUC": grid_rf.best_score_,
        "Best Parameters": grid_rf.best_params_
    })

tuned_results_df = pd.DataFrame(tuned_results)
tuned_results_df.sort_values("Best CV AUC", ascending=False).round(4)

,Model,Best CV AUC,Best Parameters
1,RandomForest,0.9154,"{'model__max_depth': 10, 'model__min_samples_l..."
0,LogReg,0.9091,"{'model__C': 0.1, 'model__penalty': 'l2'}"


In [57]:
final_models = {
    "LogReg": grid_lr.best_estimator_,
    "RandomForest": grid_rf.best_estimator_
}
res_final = cv_table(X_train, y_train, final_models)
print(res_final.sort_values("AUC", ascending=False))

d:\Ameng\Data Science Project\heart-failure-prediction\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
d:\Ameng\Data Science Project\heart-failure-prediction\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  war

                 AUC      F1     PRE     REC
model                                       
RandomForest  0.9154  0.7469  0.7334  0.7675
LogReg        0.9091  0.7823  0.7526  0.8175


Based on hyperparameter tuning results using 5-fold cross-validation, XGBoost was selected as the best model candidate because it yielded the highest AUC (0.9218) and Precision (0.8079), with an F1-score (0.7638) close to that of Logistic Regression (0.7823). Although Logistic Regression achieved the highest Recall (0.8175), XGBoost delivered superior overall performance based on the combination of evaluation metrics.

In [58]:
def select_recall_threshold(y_true, y_proba, target_recall=0.80):
    thresholds = np.arange(0.05, 0.96, 0.01)
    threshold_results = []

    for t in thresholds:
        y_pred = (y_proba >= t).astype(int)
        threshold_results.append({
            "threshold": t,
            "precision": precision_score(y_true, y_pred, zero_division=0),
            "recall": recall_score(y_true, y_pred, zero_division=0),
            "f1": f1_score(y_true, y_pred, zero_division=0),
            "f2": fbeta_score(y_true, y_pred, beta=2, zero_division=0)
        })

    threshold_df = pd.DataFrame(threshold_results)
    eligible = threshold_df[threshold_df["recall"] >= target_recall]

    if not eligible.empty:
        best_t = eligible.sort_values(["precision", "f2"], ascending=False).iloc[0]["threshold"]
    else:
        best_t = threshold_df.sort_values(["recall", "f2"], ascending=False).iloc[0]["threshold"]

    selected = threshold_df.iloc[(threshold_df["threshold"] - best_t).abs().argmin()].copy()

    return float(best_t), threshold_df, selected

## XGBoost-specific preprocessing experiment

In [59]:
def build_xgb_legacy_preprocessor():
    log_cols = ["creatinine_phosphokinase", "serum_creatinine", "platelets", "time"]
    scale_cols = ["age", "ejection_fraction", "serum_sodium"]
    bin_cols = ["anaemia", "diabetes", "high_blood_pressure", "sex", "smoking"]

    log_pipeline = Pipeline([
        ("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
        ("scale", StandardScaler())
    ])
    
    scale_pipeline = Pipeline([
        ("scale", StandardScaler())
    ])

    return ColumnTransformer([
        ("log", log_pipeline, log_cols),
        ("scale", scale_pipeline, scale_cols),
        ("pass", "passthrough", bin_cols)
    ], verbose_feature_names_out=False)

In [60]:
xgb_common_params = {
    "random_state": 42,
    "eval_metric": "logloss",
    "n_jobs": 1
}

xgb_param_values = {
    "n_estimators": [100, 200, 300],
    "learning_rate": [0.01, 0.05, 0.1],
    "max_depth": [2, 3, 5],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0]
}

In [61]:
xgb_variants = {
    "legacy": {
        "label": "XGBoost + old preprocessing",
        "estimator": Pipeline([
            ("pre", build_xgb_legacy_preprocessor()),
            ("model", XGBClassifier(**xgb_common_params))
        ]),
        "param_grid": {
            f"model__{key}": value
            for key, value in xgb_param_values.items()
        },
        "weight_param": "model__scale_pos_weight"
    },

    "raw": {
        "label": "XGBoost without scaling/log transform",
        "estimator": XGBClassifier(**xgb_common_params),
        "param_grid": xgb_param_values,
        "weight_param": "scale_pos_weight"
    }
}

In [62]:
target_recall = 0.80

neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
class_ratio = neg_count / pos_count

scale_pos_weight_values = sorted(set([1.0, 1.5, 2.0, round(float(class_ratio), 2), 2.5, 3.0]))

In [63]:
def run_xgb_variant(variant_key):
    config = xgb_variants[variant_key]

    grid = GridSearchCV(
        estimator=config["estimator"],
        param_grid=config["param_grid"],
        cv=cv,
        scoring="roc_auc",
        n_jobs=1,
        refit=True,
        return_train_score=False
    )

    grid.fit(X_train, y_train)

    candidate_rows = []

    for weight in scale_pos_weight_values:
        candidate_model = clone(grid.best_estimator_)

        candidate_model.set_params(
            **{config["weight_param"]: weight}
        )

        candidate_oof = cross_val_predict(
            candidate_model,
            X_train,
            y_train,
            cv=cv,
            method="predict_proba",
            n_jobs=1
        )[:, 1]

        candidate_threshold, _, candidate_selected = (
            select_recall_threshold(
                y_train,
                candidate_oof,
                target_recall=target_recall
            )
        )

        candidate_rows.append({
            "variant": config["label"],
            "scale_pos_weight": weight,
            "oof_auc": roc_auc_score(y_train, candidate_oof),
            "best_threshold": candidate_threshold,
            "oof_precision": candidate_selected["precision"],
            "oof_recall": candidate_selected["recall"],
            "oof_f1": candidate_selected["f1"],
            "oof_f2": candidate_selected["f2"]
        })

    weight_results = pd.DataFrame(candidate_rows)

    eligible = weight_results[
        weight_results["oof_recall"] >= target_recall
    ]

    if not eligible.empty:
        weight_winner = eligible.sort_values(
            ["oof_precision", "oof_f2", "oof_auc"],
            ascending=False
        ).iloc[0]
    else:
        weight_winner = weight_results.sort_values(
            ["oof_recall", "oof_f2", "oof_auc"],
            ascending=False
        ).iloc[0]

    best_weight = float(
        weight_winner["scale_pos_weight"]
    )

    selected_model = clone(grid.best_estimator_)

    selected_model.set_params(
        **{config["weight_param"]: best_weight}
    )

    selected_model.fit(X_train, y_train)

    selected_oof = cross_val_predict(
        selected_model,
        X_train,
        y_train,
        cv=cv,
        method="predict_proba",
        n_jobs=1
    )[:, 1]

    best_threshold, threshold_df, selected_oof_metrics = (
        select_recall_threshold(
            y_train,
            selected_oof,
            target_recall=target_recall
        )
    )

    test_proba = selected_model.predict_proba(X_test)[:, 1]

    test_auc = roc_auc_score(y_test, test_proba)

    y_pred_default = (test_proba >= 0.50).astype(int)
    y_pred_optimized = (
        test_proba >= best_threshold
    ).astype(int)

    summary = {
        "variant": config["label"],
        "cv_auc": grid.best_score_,
        "best_params": str(grid.best_params_),
        "scale_pos_weight": best_weight,
        "threshold": best_threshold,

        "oof_auc": roc_auc_score(y_train, selected_oof),
        "oof_precision": selected_oof_metrics["precision"],
        "oof_recall": selected_oof_metrics["recall"],
        "oof_f1": selected_oof_metrics["f1"],
        "oof_f2": selected_oof_metrics["f2"],

        "test_auc": test_auc,

        "test_f1_default": f1_score(
            y_test,
            y_pred_default,
            zero_division=0
        ),
        "test_precision_default": precision_score(
            y_test,
            y_pred_default,
            zero_division=0
        ),
        "test_recall_default": recall_score(
            y_test,
            y_pred_default,
            zero_division=0
        ),

        "test_f1_optimized": f1_score(
            y_test,
            y_pred_optimized,
            zero_division=0
        ),
        "test_precision_optimized": precision_score(
            y_test,
            y_pred_optimized,
            zero_division=0
        ),
        "test_recall_optimized": recall_score(
            y_test,
            y_pred_optimized,
            zero_division=0
        )
    }

    return {
        "grid": grid,
        "model": selected_model,
        "summary": summary,
        "weight_results": weight_results,
        "threshold_df": threshold_df,
        "test_proba": test_proba
    }

In [64]:
xgb_results = {}

for variant_key in xgb_variants:
    print(f"Running: {xgb_variants[variant_key]['label']}")
    xgb_results[variant_key] = run_xgb_variant(variant_key)


comparison_df = pd.DataFrame([
    result["summary"]
    for result in xgb_results.values()
])

weight_results_df = pd.concat([
    result["weight_results"]
    for result in xgb_results.values()
], ignore_index=True)

display(
    comparison_df.sort_values(
        ["oof_recall", "oof_precision", "oof_f2", "oof_auc"],
        ascending=False
    ).round(4)
)

Running: XGBoost + old preprocessing
Running: XGBoost without scaling/log transform


,variant,cv_auc,best_params,scale_pos_weight,threshold,oof_auc,oof_precision,oof_recall,oof_f1,oof_f2,test_auc,test_f1_default,test_precision_default,test_recall_default,test_f1_optimized,test_precision_optimized,test_recall_optimized
1,XGBoost without scaling/log transform,0.9185,"{'colsample_bytree': 0.8, 'learning_rate': 0.0...",2.0,0.48,0.9108,0.7750,0.8052,0.7898,0.7990,0.8408,0.7059,0.8000,0.6316,0.7429,0.8125,0.6842
0,XGBoost + old preprocessing,0.9209,"{'model__colsample_bytree': 0.8, 'model__learn...",2.0,0.43,0.9115,0.7294,0.8052,0.7654,0.7888,0.8870,0.7647,0.8667,0.6842,0.7778,0.8235,0.7368
